In [15]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import numpy as np

In [16]:
from Data_preparation import create_fish_pipeline, prepare_fish_data


In [17]:
def evaluate_random_forest(X_train, y_train, X_dev, y_dev):
    print("Evaluating Random Forest Regressor...")

    param_grid = {
        'algo__n_estimators': [100, 200],
        'algo__max_depth': [5, 10],
        'algo__min_samples_split': [2, 5]
    }

    pipeline = create_fish_pipeline()

    pipeline_with_algo = Pipeline(steps=[
        ('preprocessor', pipeline),
        ('algo', RandomForestRegressor(random_state=42))
    ])

    grid_search = GridSearchCV(
        pipeline_with_algo, param_grid,
        cv=5,  # 3-fold cross-validation
        scoring='r2',  # Use R² as the evaluation metric
        verbose=1  # Show progress in terminal
    )
    grid_search.fit(X_train, y_train)

    # This shows us our best model based on cross-validation R² score.
    best_estimator = grid_search.best_estimator_

    # We are making predicitons on the dev set here
    y_pred = best_estimator.predict(X_dev)

    # Here we are calculating the following values
    mse = mean_squared_error(y_dev, y_pred)
    mae = mean_absolute_error(y_dev, y_pred)
    r2 = r2_score(y_dev, y_pred)

    # Shows you the best performance from the training phase and the hyperparameters that gave it.
    print("Grid searching is done!")
    print("Best score (neg MSE):", grid_search.best_score_)
    print("Best hyperparameters:")
    print(grid_search.best_params_)

    return best_estimator, mse, mae, r2

In [18]:
# Step 1: Prepare fish data (split into train/dev/test)
X_train, X_dev, X_test, y_train, y_dev, y_test = prepare_fish_data(ratios=((1/10), (1/10)))

# Step 2: Run hyperparameter tuning on train/dev sets
best_model, dev_mse, dev_mae, dev_r2 = evaluate_random_forest(X_train, y_train, X_dev, y_dev)

print("\n----- Dev Set Performance -----")
print("Dev MSE:", dev_mse)
print("Dev MAE:", dev_mae)
print("Dev R²:", dev_r2)

# Step 3: Evaluate best model on test set
y_test_pred = best_model.predict(X_test)

test_mse = mean_squared_error(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("\n----- Test Set Performance -----")
print("Test MSE:", test_mse)
print("Test MAE:", test_mae)
print("Test R²:", test_r2)

Evaluating Random Forest Regressor...
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Grid searching is done!
Best score (neg MSE): 0.7967560577199437
Best hyperparameters:
{'algo__max_depth': 10, 'algo__min_samples_split': 2, 'algo__n_estimators': 100}

----- Dev Set Performance -----
Dev MSE: 0.0012833502028379532
Dev MAE: 0.002156727468706192
Dev R²: 0.9297057216834038

----- Test Set Performance -----
Test MSE: 0.06174652027970335
Test MAE: 0.005993580205779986
Test R²: 0.6891720607332786


## 🌲 Random Forest Regression Results

### 🔧 Best Hyperparameters (from Grid SearchCV)
- `max_depth`: 10  
- `min_samples_split`: 2  
- `n_estimators`: 100  

---

### 📊 Development (Validation) Set Performance
- **R² Score:** 0.9297  
- **Mean Absolute Error (MAE):** 0.0022  
- **Mean Squared Error (MSE):** 0.00128  

---

### 🧪 Test Set Performance
- **R² Score:** 0.6892  
- **Mean Absolute Error (MAE):** 0.00599  
- **Mean Squared Error (MSE):** 0.06175  
